# 🚀 `spec-rama` PEFT Benchmark Runner (Google Colab T4)

Este notebook permite reproducir de forma 100% limpia, trazable y autónoma los benchmarks de **`spec-rama` (EXP-11 y EXP-12)** en una GPU gratuita de Google Colab (**NVIDIA Tesla T4**).

---

### 📋 Objetivos del Benchmark:
1. **EXP-11**: Comparativa Head-to-Head rigurosa de recuperación de cuantización NF4 (SpecRAMA Wavelet 32x32 vs Standard LoRA r=4 con base 100% congelada).
2. **EXP-12**: Barrido de learning rate en LoRA con base aislada para auditar la sensibilidad y estabilidad frente a hiperparámetros.
3. **Serialización**: Generación de `exp11_results.json` y `exp12_results.json` con conteo exacto de parámetros entrenables y métricas de perplejidad / bits por token.

## 1. Configuración del Entorno y Verificación de GPU

In [ ]:
# 1. Instalar dependencias necesarias
!pip install -q torch transformers datasets numpy matplotlib pandas accelerate

# 2. Clonar repositorio si estamos en Colab y no existe localmente
import os
if not os.path.exists('spec_rama'):
    !git clone https://github.com/mcarbonell/spec-rama.git
    %cd spec-rama

# 3. Verificar GPU
import torch
print("=" * 60)
print("VERIFICACIÓN DE HARDWARE EN COLAB")
print("=" * 60)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Disponible:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name:         {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total:       {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 60)

## 2. Ejecutar EXP-11: Benchmark Head-to-Head (NF4 + SpecRAMA vs LoRA)

> **Nota**: Evalúa los 7 brazos experimentales sobre GPT-2 Small cuantizado en NF4 por bloques (`block_size=64`) con 48 capas lineales adaptadas y base estrictamente congelada.

In [ ]:
from benchmarks.exp11_proper_qlora_nf4_baseline import run_exp11

# Ejecutar EXP-11 (steps=500, max_test_samples=100 bloques = 25.6k tokens)
# Para evaluar sobre el test set completo (~287k tokens), poner full_test=True
results_exp11 = run_exp11(
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_steps=500,
    max_test_samples=100,
    full_test=False,
    save_json=True
)

## 3. Ejecutar EXP-12: Barrido de Hiperparámetros de LoRA (Base Aislada)

> **Nota**: Ejecuta el barrido de LR sobre LoRA ($r=4, alpha=8.0$) en una base NF4 con los embeddings y LayerNorms 100% congelados (589.824 parámetros entrenables).

In [ ]:
from benchmarks.exp12_lora_hyperparameter_sweep import run_exp12

results_exp12 = run_exp12(
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_steps=500,
    max_test_samples=100,
    full_test=False,
    save_json=True
)

## 4. Visualización de Resultados y Tabla Comparativa

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json

# Cargar resultados de EXP-11
with open('benchmarks/exp11_results.json', 'r') as f:
    data11 = json.load(f)

df11 = pd.DataFrame(data11['results'])
display(df11[['arm', 'name', 'format', 'adapter_size', 'trainable_params', 'test_ppl', 'bits_per_token', 'delta_bpt_vs_native', 'status']])

# Gráfico comparativo de PPL vs Parámetros Entrenables
plt.figure(figsize=(10, 5), dpi=120)
tuned_arms = df11[df11['trainable_params'] > 0]

plt.bar(tuned_arms['name'], tuned_arms['test_ppl'], color=['#3b82f6', '#10b981', '#f59e0b', '#8b5cf6'])
plt.axhline(df11.loc[df11['arm']==1, 'test_ppl'].values[0], color='red', linestyle='--', label='FP32 Native (Zero-Shot)')
plt.axhline(df11.loc[df11['arm']==3, 'test_ppl'].values[0], color='gray', linestyle=':', label='NF4 Base (Zero-Shot)')

plt.title('Comparativa de Perplejidad en WikiText-2 (Menor es Mejor)', fontsize=12, fontweight='bold')
plt.ylabel('Test Perplexity (PPL)')
plt.xticks(rotation=20, ha='right')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Descargar Resultados JSON

In [ ]:
try:
    from google.colab import files
    files.download('benchmarks/exp11_results.json')
    files.download('benchmarks/exp12_results.json')
    print(" Archivos descargados con éxito.")
except Exception as e:
    print(f"Resultados guardados localmente en: benchmarks/exp11_results.json y exp12_results.json ({e})")